This notebook perform webscraping on the Dan Murphy's site to scrape the hrefs of the wines

In [34]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException, StaleElementReferenceException
from webdriver_manager.chrome import ChromeDriverManager
import re
import time
import pandas as pd

In [35]:
# Select the colour of wines to scrape
wine_type = "Red"
#wine_type = "White"
#wine_type = "Champagne-Sparkling"

In [36]:
# Create options object
options = Options()

#options.add_argument('--headless')
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')
options.add_argument('--disable-infobars')
options.add_argument('--disable-extensions')
options.add_argument('--disable-notifications')
options.add_argument('--window-size=1724,968')
user_agent = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/96.0.4664.110 Safari/537.36"
options.add_argument(f'user-agent={user_agent}')
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
driver.get("https://www.danmurphys.com.au/")

In [37]:
# Pass home screen location pop up

# Wait for the button to be clickable
wait = WebDriverWait(driver, 10)
button = wait.until(EC.element_to_be_clickable((By.ID, "cro-activity-location-dismissal")))

# Click the button
button.click()

In [38]:
# Get all necessary Wine nav buttons
wine_nav_buttons = driver.find_elements(By.CLASS_NAME, "btn-secondary-nav")[:4]

# Get "Shop all" buttons
## Note the site has changed the button to "Browse All"
all_wine_links = []
all_wine_links.append(driver.find_element(By.XPATH, "//a[text()='Browse all' and @href='/red-wine/all']"))
all_wine_links.append(driver.find_element(By.XPATH, "//a[text()='Shop All' and @href='/white-wine/all']"))
#all_wine_links.append(driver.find_element(By.XPATH, "//a[text()='Shop all' and @href='/red-wine/rose']"))
all_wine_links.append(driver.find_element(By.XPATH, "//a[text()='Shop All' and @href='/champagne-sparkling/all']"))

In [39]:
def click_button(id):
    # Wait for the button to be clickable
    wait = WebDriverWait(driver, 10)
    button = wait.until(EC.element_to_be_clickable((By.ID, id)))
    button.click()

def click_button_by_partial_id(partial_id):
    # Wait for the button to be clickable
    wait = WebDriverWait(driver, 10)
    button = wait.until(EC.element_to_be_clickable((By.XPATH, f"//button[contains(@id, '{partial_id}')]")))
    #button = driver.find_element(By.XPATH, f"//button[contains(@id, '{partial_id}')]")
    button.click()

# click wine menu
click_button("megaNavDropdownWine")

#click wine type
if wine_type != "Champagne-Sparkling":
    click_button(f"megaNavDropdown{wine_type}Wine")
else:
    #click_button("megaNavDropdownChampagne&SparklingWine")
    click_button_by_partial_id("megaNavDropdownChampagne")

In [44]:
element_html = all_wine_links[0].get_attribute('outerHTML')
element_html

'<a _ngcontent-yvx-c32="" class="tertiary-links-more ng-star-inserted" href="/red-wine/all">Browse all</a>'

In [40]:
# click on all red wines button
if wine_type == "Red":
    all_wine_links[0].click()
elif wine_type == "White":element_html = element.get_attribute('outerHTML')
    all_wine_links[1].click()
else:
    all_wine_links[2].click()

In [31]:
# show all products

## get show more button
#show_more_button = driver.find_element(By.CLASS_NAME, "dans-button")

def get_load_summry():
    loader_summary = driver.find_element(By.CLASS_NAME, "infinite-loader__summary") ## get summary of loaded products
    pattern = r'\d+'
    return re.findall(pattern, loader_summary.text) ## Extract numbers using regular expression

def all_loaded():
    load_summary = get_load_summry()
    print(load_summary[0], "loaded out of", load_summary[1])
    if int(load_summary[0]) < int(load_summary[1]):
        return False
    else:
        return True

retried = 0
max_retries = 5
while retried < max_retries:
    try:
        while not all_loaded():
            wait = WebDriverWait(driver, 5)
            show_more_button = wait.until(EC.element_to_be_clickable((By.CLASS_NAME, "dans-button")))
            show_more_button.click()
            time.sleep(5)
        break
    except TimeoutException:
        print("Timeout occurred while waiting for show more button.")
        time.sleep(5)
        retried += 1
    except NoSuchElementException:
        print("Loader summary not found. Exiting the loop.")
        time.sleep(5)
        retried += 1
    except StaleElementReferenceException:
        print("Stale element reference encountered.")
        time.sleep(5)
        retried += 1


Loader summary not found. Exiting the loop.
Loader summary not found. Exiting the loop.
Loader summary not found. Exiting the loop.
Loader summary not found. Exiting the loop.
Loader summary not found. Exiting the loop.


In [ ]:
# get all product links
product_list_ul = driver.find_element(By.CLASS_NAME, "search-results__product-list")
product_links = product_list_ul.find_elements(By.XPATH, ".//a[contains(@class, 'ng-tns') and contains(@href, '/product/')]")

In [ ]:
len(product_links)

In [ ]:
# save links to products
wine_product_links = []
for i in range(len(product_links)):
    print("extracting", i+1, "of", len(product_links), "links")
    wine_product_links.append(product_links[i].get_attribute("href"))

In [ ]:
wine_product_links_df = {"href" : wine_product_links, "type" : wine_type}

wine_product_links_df = pd.DataFrame(wine_product_links_df)

file_path = f"data/{wine_type}_wine_hrefs.csv"
wine_product_links_df.to_csv(file_path, index=False)